# Tuần 3 — Huấn luyện `ViT-LoRA (r=8, α=16)`, Threshold Moving & Đánh giá Hộp giới hạn Tự sinh trên Dataset 1 & Dataset 2

**Thành viên phụ trách:** Thái Văn Huy

### Hai nhiệm vụ thực nghiệm trên Dataset 1 (Navoneel — 253 ảnh) và Dataset 2 (indk214 — 9.215 ảnh):
1. **Nhiệm vụ 1:** Huấn luyện **`ViT-LoRA (r=8, α=16)`** trên **Dataset 1 (5-Fold CV)** và **Dataset 2 (phân loại 4 lớp + hồi quy bounding box)**; thực hiện tinh chỉnh ngưỡng quyết định xác suất (**Threshold Moving** từ `0.50` xuống `0.35–0.40`, chọn hoàn toàn trên tập validation) để đẩy Độ nhạy (**Recall**) lớp có u vượt mốc **96%**.
2. **Nhiệm vụ 2:** Đánh giá chất lượng hộp giới hạn tự sinh trên tập kiểm thử phân đoạn độc lập của **Dataset 2 (`329` cặp ảnh–mask, không dùng GT prompt)** bằng chỉ số **`Mean Box IoU`** và tỷ lệ bắt trúng tâm khối u (**`Pointing Game Accuracy`**).

---
## Cell 1: Mount Google Drive, Cấu hình Đường dẫn & Hàm Hỗ trợ Thực thi

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

RUNTIME_MODE = "local" if os.name == "nt" else "colab"
_cwd = Path.cwd().resolve()
LOCAL_PROJECT_DIR = _cwd.parent if _cwd.name == "notebooks" else _cwd
DRIVE_SYNC_DIR = None  # Local: Path(r"duong_dan_Google_Drive\My Drive\Huy\tuan3")

PROJECT_DIR = LOCAL_PROJECT_DIR if RUNTIME_MODE == "local" else Path("/content/BrainTumorDetect")
RUN_DIR = PROJECT_DIR / "runs/huy_week3" if RUNTIME_MODE == "local" else Path("/content/drive/MyDrive/Huy/tuan3")
SOURCE_MODE = "existing" if RUNTIME_MODE == "local" else "github"  # "github" | "archive" | "existing"
DATA_MODE = "existing" if RUNTIME_MODE == "local" else "kaggle"    # "kaggle" | "drive_zip" | "existing"
REPO_URL = "https://github.com/MikanQuyt/BrainTumorDetect.git"
REPO_REF = "dev"
NAVONEEL_ZIP = ""
INDK214_ZIP = ""

# Siêu tham số huấn luyện
MAX_EPOCHS = 1000
EARLY_STOPPING_PATIENCE = 25
BATCH_SIZE = 8
NUM_WORKERS = 0 if RUNTIME_MODE == "local" else 2
DS1_FOLD = None
RUN_DS1 = True
RUN_DS2 = True
RUN_OFFLINE_TESTS = True
PREPARE_DATA = RUNTIME_MODE != "local"
INSTALL_REQUIREMENTS = RUNTIME_MODE != "local"
UNINSTALL_TORCHAO = RUNTIME_MODE != "local"
REQUIRE_DS2_CHECKPOINT = False
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_synced_files = {}


def sync_outputs():
    if RUNTIME_MODE != "local" or DRIVE_SYNC_DIR is None:
        return
    for source in sorted(RUN_DIR.rglob("*")):
        if not source.is_file() or source.suffix not in {".pth", ".json", ".csv", ".png", ".txt"}:
            continue
        relative = source.relative_to(RUN_DIR)
        stat = source.stat()
        signature = (stat.st_mtime_ns, stat.st_size)
        if _synced_files.get(relative) == signature:
            continue
        destination = Path(DRIVE_SYNC_DIR) / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(f".{destination.name}.{os.getpid()}.tmp")
        try:
            shutil.copy2(source, temporary)
            current = source.stat()
            if signature != (current.st_mtime_ns, current.st_size):
                continue
            os.replace(temporary, destination)
            _synced_files[relative] = signature
        finally:
            temporary.unlink(missing_ok=True)


def try_sync_outputs():
    try:
        sync_outputs()
    except OSError as error:
        print(f"Drive sync pending: {error}. Checkpoint gốc vẫn ở {RUN_DIR}")


def run_command(command, *, cwd=None, log_name=None):
    log = None
    if log_name is not None:
        log_dir = RUN_DIR / "logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        log = (log_dir / Path(log_name).name).open("a", encoding="utf-8")
        log.write("\nCOMMAND: " + " ".join(map(str, command)) + "\n")
        log.flush()
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    try:
        with subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    if log is not None:
                        log.write(line)
                        log.flush()
                    if " patience=" in line or line.startswith("RESUME "):
                        try_sync_outputs()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if status:
            raise subprocess.CalledProcessError(status, command)
    finally:
        if log is not None:
            log.close()
        try_sync_outputs()


if RUNTIME_MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise FileNotFoundError("Google Drive chưa được mount thành công tại /content/drive/MyDrive")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
elif RUNTIME_MODE == "local":
    if not (PROJECT_DIR / "engine/train_vit_lora_huy.py").is_file():
        raise FileNotFoundError(f"Sai LOCAL_PROJECT_DIR: {PROJECT_DIR}")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    if DRIVE_SYNC_DIR is not None:
        DRIVE_SYNC_DIR = Path(DRIVE_SYNC_DIR).resolve()
        local = RUN_DIR.resolve()
        DRIVE_SYNC_DIR.mkdir(parents=True, exist_ok=True)
        if (
            DRIVE_SYNC_DIR == local
            or DRIVE_SYNC_DIR.is_relative_to(local)
            or local.is_relative_to(DRIVE_SYNC_DIR)
        ):
            raise ValueError("RUN_DIR và DRIVE_SYNC_DIR phải là hai thư mục tách biệt")
else:
    raise ValueError("RUNTIME_MODE phải là local hoặc colab")

print(f"Runtime: {RUNTIME_MODE} | Source: {SOURCE_MODE} | Python: {sys.executable}")
print(f"Checkpoint và kết quả: {RUN_DIR}")

---
## Cell 2: Nạp Mã nguồn, Cài đặt Thư viện & Kiểm tra GPU

In [ ]:
if SOURCE_MODE == "existing":
    pass
elif SOURCE_MODE == "archive":
    source_dir = RUN_DIR / "source"
    source_dir.mkdir(exist_ok=True)
    archive = source_dir / "BrainTumorDetect_Huy_week3.zip"
    if not archive.is_file():
        from google.colab import files
        uploaded = files.upload()
        expected = "BrainTumorDetect_Huy_week3.zip"
        if expected not in uploaded:
            raise ValueError(f"Expected source archive: {expected}")
        archive.write_bytes(uploaded[expected])
        del uploaded
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            target = (PROJECT_DIR.parent / entry.filename).resolve()
            if not target.is_relative_to(PROJECT_DIR.resolve()):
                raise ValueError("Gói source phải chỉ chứa thư mục BrainTumorDetect/")
        z.extractall(PROJECT_DIR.parent)
elif SOURCE_MODE == "github":
    if not (PROJECT_DIR / ".git").exists():
        run_command(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT_DIR)])
    else:
        run_command(["git", "fetch", "origin", REPO_REF], cwd=PROJECT_DIR)
        run_command(["git", "switch", REPO_REF], cwd=PROJECT_DIR)
        run_command(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=PROJECT_DIR)
else:
    raise ValueError("SOURCE_MODE phải là existing, archive hoặc github")

required = [
    "engine/train_vit_lora_huy.py",
    "data/prepare_huy_week3.py",
    "reports/tuan2/vit_baseline/ds1/splits.csv",
    "reports/tuan2/vit_baseline/ds2/splits.csv",
]
for name in required:
    if not (PROJECT_DIR / name).is_file():
        raise FileNotFoundError(f"Thiếu {name}. Kiểm tra gói source/nhánh Git.")
os.chdir(PROJECT_DIR)

if INSTALL_REQUIREMENTS:
    if UNINSTALL_TORCHAO:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    try:
        run_command([sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/requirements-huy-week3.txt"])
    except subprocess.CalledProcessError as exc:
        print(f"Cảnh báo: cài đặt ghim phiên bản gặp xung đột ({exc}); chuyển sang phiên bản tương thích...")
        run_command([
            sys.executable, "-m", "pip", "install", "-q",
            "transformers>=4.46", "peft>=0.14", "accelerate>=1.0",
            "huggingface-hub>=0.26", "albumentations>=2.0",
            "opencv-python-headless", "scikit-learn", "timm",
            "matplotlib", "kagglehub", "nbformat",
        ])

check = (
    "import torch, transformers, peft, json, sys, platform; "
    "from models.vit_lora import ViTLoRA; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print(json.dumps(dict(python=sys.version, executable=sys.executable, "
    "platform=platform.platform(), torch=torch.__version__, "
    "transformers=transformers.__version__, peft=peft.__version__, "
    "cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))))"
)
env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONPATH=str(PROJECT_DIR))
runtime_info = json.loads(subprocess.check_output([sys.executable, "-c", check], text=True, encoding="utf-8", env=env))
print(
    f"Torch: {runtime_info['torch']} | Transformers: {runtime_info['transformers']} | "
    f"PEFT: {runtime_info['peft']} | CUDA: {runtime_info['cuda']} | GPU: {runtime_info['gpu']}"
)
environment_dir = RUN_DIR / "environment"
environment_dir.mkdir(exist_ok=True)
(environment_dir / f"{RUNTIME_MODE}_runtime.json").write_text(
    json.dumps(runtime_info, ensure_ascii=False, indent=2), encoding="utf-8"
)
(environment_dir / f"pip-freeze-{RUNTIME_MODE}.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True, encoding="utf-8", env=env),
    encoding="utf-8",
)

---
## Cell 3: Kiểm định Logic Pipeline (`tests.test_huy_week3`)
Xác minh tự động split chống rò rỉ, bảo toàn mask khi crop, chọn ngưỡng trên validation, early stopping, checkpoint resume và đánh giá Auto-Prompt.

In [ ]:
if RUN_OFFLINE_TESTS:
    run_command([sys.executable, "-m", "unittest", "tests.test_huy_week3", "-v"], log_name="pipeline_tests.log")

---
## Cell 4: Tải Dữ liệu Gốc — Dataset 1 (`Navoneel`, 253 ảnh) & Dataset 2 (`indk214`, 9.215 ảnh)

In [ ]:
raw = PROJECT_DIR / "dataset"
raw.mkdir(exist_ok=True)


def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if not (destination / entry.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Đường dẫn ZIP không hợp lệ")
        z.extractall(destination)
    return destination


if DATA_MODE == "existing":
    for name in ("audit.json", "splits.json", "ds1_annotations.json", "ds2_annotations.json"):
        if not (raw / "huy_week3" / name).is_file():
            raise FileNotFoundError(f"Thiếu dữ liệu đã chuẩn bị: {raw / 'huy_week3' / name}")
    print("Dùng lại dữ liệu đã chuẩn bị:", raw / "huy_week3")
else:
    if DATA_MODE == "kaggle":
        import kagglehub
        nav_source = Path(kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection"))
        ind_source = Path(kagglehub.dataset_download("indk214/brain-tumor-dataset-segmentation-and-classification"))
    elif DATA_MODE == "drive_zip":
        if not Path(NAVONEEL_ZIP).is_file() or not Path(INDK214_ZIP).is_file():
            raise FileNotFoundError("Điền đúng hai đường dẫn ZIP trên Drive (NAVONEEL_ZIP và INDK214_ZIP)")
        nav_source = safe_extract(NAVONEEL_ZIP, Path("/content/raw_navoneel"))
        ind_source = safe_extract(INDK214_ZIP, Path("/content/raw_indk214"))
    else:
        raise ValueError("DATA_MODE phải là existing, kaggle hoặc drive_zip")

    nav_candidates = [nav_source] + sorted(p.parent for p in nav_source.rglob("yes") if p.is_dir())
    nav_base = next((p for p in nav_candidates if (p / "yes").is_dir() and (p / "no").is_dir()), None)
    ind_candidates = [ind_source / "DATASET"] + sorted(p for p in ind_source.rglob("DATASET") if p.is_dir())
    ind_base = next((p for p in ind_candidates if (p / "classification").is_dir() and (p / "Segmentation").is_dir()), None)
    if nav_base is None or ind_base is None:
        raise ValueError("Bố cục dataset khác chuẩn tuần 2; kiểm tra thư mục trước khi tiếp tục")
    for label in ("no", "yes"):
        shutil.copytree(nav_base / label, raw / "navoneel" / label, dirs_exist_ok=True)
    shutil.copytree(ind_base, raw / "indk214" / "DATASET", dirs_exist_ok=True)
    print("Đã lấy dữ liệu gốc DS1 & DS2 thành công.")

---
## Cell 5: Tiền xử lý, Kiểm toán Trùng lặp Mức Pixel & Chia tập (`data.prepare_huy_week3`)
- DS1 giữ nguyên 5 fold tuần 2.
- DS2 mở rộng nhóm trùng theo ảnh gốc + ảnh sau crop/CLAHE và cầu nối segmentation, sửa xung đột theo ưu tiên **`test > validation > train`** (bảo toàn 100% ảnh test cũ).
- Mask được crop đồng bộ với MRI và kiểm tra bảo toàn pixel khối u.

In [ ]:
if PREPARE_DATA:
    run_command([sys.executable, "-m", "data.prepare_huy_week3"], log_name="prepare_data.log")

audit = json.loads((PROJECT_DIR / "dataset/huy_week3/audit.json").read_text(encoding="utf-8"))
for dataset in ("ds1", "ds2"):
    print(f"{dataset.upper()} | images: {audit[dataset]['n']} | masks: {audit[dataset]['n_masks']}")

for source, destination in (
    ("audit.json", "data_audit.json"),
    ("splits.json", "data_splits.json"),
    ("ds2_split_changes.csv", "ds2_split_changes.csv"),
):
    shutil.copy2(PROJECT_DIR / "dataset/huy_week3" / source, RUN_DIR / destination)

---
## Cell 6: Thí nghiệm 1 — Huấn luyện `ViT-LoRA (r=8, α=16)` & Threshold Moving trên Dataset 1 (5-Fold CV)
- Đóng băng backbone `ViT-Base/16`, cập nhật LoRA + head bằng `AdamW`, `ReduceLROnPlateau`, early stopping theo validation loss (`patience=25`), tự động `--resume auto` theo từng epoch.
- Ngưỡng trong `0.35–0.40` được chọn riêng trên tập validation của từng fold rồi cố định khi chấm outer-test.

In [ ]:
if RUNTIME_MODE == "local":
    import psutil
    for process in psutil.process_iter(["pid", "cmdline", "cwd"]):
        try:
            command = process.info["cmdline"] or []
            if "engine.train_vit_lora_huy" not in command or "--out" not in command:
                continue
            active_out = Path(command[command.index("--out") + 1])
            if not active_out.is_absolute():
                if not process.info["cwd"]:
                    continue
                active_out = Path(process.info["cwd"]) / active_out
            if active_out.resolve() == RUN_DIR.resolve():
                raise RuntimeError(f"Tiến trình khác đang train vào {RUN_DIR} (PID {process.pid}).")
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue

try_sync_outputs()

base_command = [
    sys.executable, "-m", "engine.train_vit_lora_huy",
    "--out", str(RUN_DIR),
    "--epochs", str(MAX_EPOCHS),
    "--patience", str(EARLY_STOPPING_PATIENCE),
    "--batch-size", str(BATCH_SIZE),
    "--workers", str(NUM_WORKERS),
    "--device", "cuda", "--amp", "--resume", "auto",
]

if RUN_DS1:
    command = base_command + ["--dataset", "ds1"]
    if DS1_FOLD is not None:
        command += ["--fold", str(DS1_FOLD)]
    run_command(command, log_name="train_ds1.log")

---
## Cell 7: Thí nghiệm 2 — Huấn luyện `ViT-LoRA (r=8, α=16)` + Box Head, Threshold Moving & Đánh giá Hộp tự sinh trên Dataset 2
- Huấn luyện đồng thời nhánh phân loại 4 lớp (`no_tumor`, `glioma`, `meningioma`, `pituitary`) và nhánh hồi quy hộp (`ViTDetBoxHead`).
- Đánh giá Threshold Moving trên classification test (`1.032` ảnh) và đánh giá **`Mean Box IoU`** + **`Pointing Game Accuracy`** trên segmentation test độc lập (`329` ảnh, không GT prompt).

In [ ]:
if RUN_DS2 and REQUIRE_DS2_CHECKPOINT:
    latest = RUN_DIR / "ds2/fold_01/loss_latest.pth"
    if not latest.is_file():
        raise FileNotFoundError(f"Thiếu checkpoint để resume: {latest}")
    check_ep = (
        "import torch, sys; p=torch.load(sys.argv[1], map_location='cpu', weights_only=False); "
        "print('Checkpoint epoch:', p['epoch'], '| Next epoch:', p['epoch']+1)"
    )
    run_command([sys.executable, "-c", check_ep, str(latest)])

if RUN_DS2:
    run_command(base_command + ["--dataset", "ds2"], log_name="train_ds2.log")

---
## Cell 8: Tổng hợp Kết quả Thực nghiệm (DS1 5-Fold CV, DS2 Phân loại, Threshold Moving & Định vị Hộp tự sinh)

In [ ]:
from IPython.display import Image, display


def _resolve_report(relative):
    candidates = [RUN_DIR / relative]
    if RUNTIME_MODE == "local" and DRIVE_SYNC_DIR is not None:
        candidates.append(Path(DRIVE_SYNC_DIR) / relative)
    return next((p for p in candidates if p.is_file()), None), candidates


ds1_path, ds1_cands = _resolve_report(Path("ds1/cv_summary.json"))
if ds1_path is not None:
    ds1 = json.loads(ds1_path.read_text(encoding="utf-8"))
    d050 = ds1["detection_050_pooled"]
    dsel = ds1["detection_selected_pooled"]
    print("=== DS1 (5-Fold CV, n=253) ===", ds1_path)
    print(
        f"  tau=0.50       : Acc={d050['accuracy']*100:.2f}% | Recall={d050['recall']*100:.2f}% "
        f"| Spec={d050['specificity']*100:.2f}% | FN={d050['fn']} | FP={d050['fp']}"
    )
    print(
        f"  Threshold Move : Acc={dsel['accuracy']*100:.2f}% | Recall={dsel['recall']*100:.2f}% "
        f"(std={ds1['fold_recall_sample_std']*100:.2f}%) | Spec={dsel['specificity']*100:.2f}% "
        f"| FN={dsel['fn']} | FP={dsel['fp']} | target>96%={ds1['target_met']}"
    )
    print(f"  Per-fold taus  : {ds1['per_fold_thresholds']}\n")
else:
    print("DS1 chưa có báo cáo tại:", *ds1_cands)

ds2_path, ds2_cands = _resolve_report(Path("ds2/fold_01/test_metrics.json"))
if ds2_path is not None:
    ds2 = json.loads(ds2_path.read_text(encoding="utf-8"))
    cls = ds2["classification"]
    d050, dsel = ds2["detection_050"], ds2["detection_selected"]
    loc = ds2["localization"]
    print("=== DS2 (Classification + Threshold Moving + Localization) ===", ds2_path)
    print(
        f"  4-Class Test (n={cls['n']}): Acc={cls['accuracy']*100:.2f}% | Macro F1={cls['f1']*100:.2f}% "
        f"| Macro Recall={cls['recall']*100:.2f}% | Macro ROC-AUC={cls['auc']*100:.2f}%"
    )
    print(
        f"  Tumor Detect (tau=0.50)       : Recall={d050['recall']*100:.2f}% | Spec={d050['specificity']*100:.2f}% "
        f"| FN={d050['fn']} | FP={d050['fp']}"
    )
    print(
        f"  Tumor Detect (tau={dsel['threshold']:.2f})       : Recall={dsel['recall']*100:.2f}% "
        f"| Spec={dsel['specificity']*100:.2f}% | FN={dsel['fn']} | FP={dsel['fp']} | target>96%={ds2['test_target_met']}"
    )
    ov = loc["overall"]
    print(
        f"  Localization (n={ov['n']}, det={ov['n_detected']}): "
        f"Mean Box IoU={ov['mean_box_iou']:.4f} | Pointing Game Acc={ov['pointing_game_accuracy']*100:.2f}%"
    )
    for cname, stats in loc["per_class"].items():
        print(
            f"    - {cname:10s} (n={stats['n']:3d}): Box IoU={stats['mean_box_iou']:.4f} "
            f"| Pointing Game={stats['pointing_game_accuracy']*100:.2f}%"
        )
    print()
else:
    print("DS2 chưa có báo cáo tại:", *ds2_cands)

for path in sorted(RUN_DIR.glob("ds*/fold_*/learning_curves.png")):
    print(path.parent)
    display(Image(filename=str(path)))

---
## Cell 9: Đồng bộ & Đóng gói Báo cáo Bàn giao (`.zip` không gồm checkpoint `.pth`)

In [ ]:
try_sync_outputs()

zip_path = RUN_DIR / "week3_ds1_ds2_huy_results.zip"
exported = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file() and p.suffix in {".json", ".csv", ".png", ".txt"} and p != zip_path:
            zf.write(p, p.relative_to(RUN_DIR))
            exported += 1

print(f"Đã đóng gói {exported} file kết quả vào: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)")
if RUNTIME_MODE == "colab" and exported > 0:
    from google.colab import files
    files.download(str(zip_path))